# Sistemas Inteligentes I
## Búsqueda adversarial: algoritmo Minimax

**Autor:** Jairo I. Vélez B.

---

# 1. Idea central: buscar cuando existe un adversario

En BFS, DFS o A* buscamos una ruta hacia un objetivo.  
En un juego ocurre algo diferente:

> después de que nosotros elegimos una acción, **otro jugador también elige**.

Por tanto, no basta con preguntarnos:

> ¿Cuál es la mejor jugada que puedo hacer?

También debemos considerar:

> ¿Qué hará mi oponente si intenta perjudicarme?

Minimax modela dos jugadores:

- **MAX:** intenta obtener el valor más alto posible.
- **MIN:** intenta obtener el valor más bajo posible.

Supondremos inicialmente que ambos jugadores actúan de manera racional.

# 2. Elementos de un problema adversarial

Un juego puede describirse mediante:

- **Estado:** configuración actual del juego.
- **Jugador actual:** indica quién debe mover.
- **Acciones:** movimientos legales desde el estado actual.
- **Resultado:** estado que se obtiene al aplicar una acción.
- **Estado terminal:** posición en la que la partida ha terminado.
- **Utilidad:** valor numérico asociado al resultado final.

Una convención sencilla puede ser:

| Resultado para MAX | Utilidad |
|---|---:|
| Victoria | `+1` |
| Empate | `0` |
| Derrota | `-1` |

En ejemplos más generales, la utilidad puede tomar cualquier valor numérico.

# 3. Primer árbol de juego

Consideremos el siguiente árbol.

- La raíz `A` pertenece a **MAX**.
- En el siguiente nivel juega **MIN**.
- Las hojas contienen valores de utilidad.

```text
                    A  (MAX)
              /         |         \
          B (MIN)    C (MIN)    D (MIN)
          / | \       / | \       / | \
         3  5  2     9  1  4     6  7  8
```

Pregunta:

> Si MIN juega racionalmente, ¿qué opción debería escoger MAX desde `A`?

In [1]:
arbol = {
    "A": ["B", "C", "D"],
    "B": ["B1", "B2", "B3"],
    "C": ["C1", "C2", "C3"],
    "D": ["D1", "D2", "D3"],
}

utilidades = {
    "B1": 3, "B2": 5, "B3": 2,
    "C1": 9, "C2": 1, "C3": 4,
    "D1": 6, "D2": 7, "D3": 8,
}

arbol, utilidades

({'A': ['B', 'C', 'D'],
  'B': ['B1', 'B2', 'B3'],
  'C': ['C1', 'C2', 'C3'],
  'D': ['D1', 'D2', 'D3']},
 {'B1': 3,
  'B2': 5,
  'B3': 2,
  'C1': 9,
  'C2': 1,
  'C3': 4,
  'D1': 6,
  'D2': 7,
  'D3': 8})

# 4. Razonamiento antes del algoritmo

Analicemos primero cada nodo de MIN.

### Nodo B

MIN puede elegir entre:

$$3,\ 5,\ 2$$

Por tanto:

$$\min(3,5,2)=2$$

### Nodo C

$$\min(9,1,4)=1$$

### Nodo D

$$\min(6,7,8)=6$$

La raíz pertenece a MAX, así que compara:

$$\max(2,1,6)=6$$

Por tanto, MAX debería elegir la rama `D`.

Esta idea es exactamente la que implementa **Minimax**.

# 5. Algoritmo Minimax

La definición recursiva puede escribirse como:

$$
V(s)=
\begin{cases}
U(s) & \text{si }s\text{ es terminal}\\
\max_{s' \in Sucesores(s)} V(s') & \text{si juega MAX}\\
\min_{s' \in Sucesores(s)} V(s') & \text{si juega MIN}
\end{cases}
$$

La recursión baja hasta los estados terminales y después los valores se
**propagan hacia arriba**.

In [2]:
def minimax(nodo, es_max, arbol, utilidades):
    # Caso base: nodo terminal
    if nodo in utilidades:
        return utilidades[nodo]

    valores = []

    for hijo in arbol[nodo]:
        valor = minimax(hijo, not es_max, arbol, utilidades)
        valores.append(valor)

    if es_max:
        return max(valores)
    else:
        return min(valores)


valor_raiz = minimax("A", True, arbol, utilidades)
valor_raiz

6

## 5.1 Obtener también la mejor jugada

Conocer el valor del estado es útil, pero normalmente necesitamos además saber:

> **¿qué acción debe ejecutar el jugador?**

La siguiente función devuelve el valor Minimax y el hijo seleccionado.

In [3]:
def mejor_jugada_minimax(nodo, es_max, arbol, utilidades):
    if nodo in utilidades:
        return utilidades[nodo], None

    opciones = []

    for hijo in arbol[nodo]:
        valor = minimax(hijo, not es_max, arbol, utilidades)
        opciones.append((valor, hijo))

    if es_max:
        valor, hijo = max(opciones, key=lambda x: x[0])
    else:
        valor, hijo = min(opciones, key=lambda x: x[0])

    return valor, hijo


valor, jugada = mejor_jugada_minimax("A", True, arbol, utilidades)

print("Valor Minimax:", valor)
print("Mejor jugada para MAX:", jugada)

Valor Minimax: 6
Mejor jugada para MAX: D


# 6. Minimax paso a paso

Para comprender mejor el algoritmo observaremos la recursión.

La sangría permite identificar la profundidad en el árbol.

In [4]:
def minimax_debug(nodo, es_max, arbol, utilidades, profundidad=0):
    sangria = "    " * profundidad
    jugador = "MAX" if es_max else "MIN"

    if nodo in utilidades:
        print(f"{sangria}{nodo}: terminal -> utilidad {utilidades[nodo]}")
        return utilidades[nodo]

    print(f"{sangria}{nodo}: turno de {jugador}")
    valores = []

    for hijo in arbol[nodo]:
        valor = minimax_debug(
            hijo,
            not es_max,
            arbol,
            utilidades,
            profundidad + 1
        )
        valores.append(valor)

    if es_max:
        resultado = max(valores)
    else:
        resultado = min(valores)

    print(f"{sangria}{nodo}: {jugador} selecciona {resultado}")
    return resultado


minimax_debug("A", True, arbol, utilidades)

A: turno de MAX
    B: turno de MIN
        B1: terminal -> utilidad 3
        B2: terminal -> utilidad 5
        B3: terminal -> utilidad 2
    B: MIN selecciona 2
    C: turno de MIN
        C1: terminal -> utilidad 9
        C2: terminal -> utilidad 1
        C3: terminal -> utilidad 4
    C: MIN selecciona 1
    D: turno de MIN
        D1: terminal -> utilidad 6
        D2: terminal -> utilidad 7
        D3: terminal -> utilidad 8
    D: MIN selecciona 6
A: MAX selecciona 6


6

### Preguntas de análisis

1. ¿Por qué MAX no selecciona directamente la hoja con valor `9`?
2. ¿Qué supone Minimax sobre el comportamiento del adversario?
3. ¿Qué ocurriría si MIN no escogiera siempre la opción de menor valor?
4. ¿Por qué los valores se calculan desde las hojas hacia la raíz?
5. ¿El valor de una hoja representa necesariamente una puntuación real del juego?

# 7. Un árbol con mayor profundidad

Ahora utilizaremos un árbol de tres decisiones.

```text
MAX → MIN → MAX → utilidad
```

Esto permite observar que los roles se alternan en cada nivel.

In [5]:
arbol_profundo = {
    "A": ["B", "C"],
    "B": ["D", "E"],
    "C": ["F", "G"],
    "D": ["D1", "D2"],
    "E": ["E1", "E2"],
    "F": ["F1", "F2"],
    "G": ["G1", "G2"],
}

utilidades_profundo = {
    "D1": 3, "D2": 5,
    "E1": 6, "E2": 9,
    "F1": 1, "F2": 2,
    "G1": 0, "G2": -1,
}

minimax_debug("A", True, arbol_profundo, utilidades_profundo)

A: turno de MAX
    B: turno de MIN
        D: turno de MAX
            D1: terminal -> utilidad 3
            D2: terminal -> utilidad 5
        D: MAX selecciona 5
        E: turno de MAX
            E1: terminal -> utilidad 6
            E2: terminal -> utilidad 9
        E: MAX selecciona 9
    B: MIN selecciona 5
    C: turno de MIN
        F: turno de MAX
            F1: terminal -> utilidad 1
            F2: terminal -> utilidad 2
        F: MAX selecciona 2
        G: turno de MAX
            G1: terminal -> utilidad 0
            G2: terminal -> utilidad -1
        G: MAX selecciona 0
    C: MIN selecciona 0
A: MAX selecciona 5


5

## 7.1 Contar nodos evaluados

En árboles pequeños Minimax resulta sencillo.  
Sin embargo, el número de posiciones posibles puede crecer rápidamente.

Contaremos cuántos nodos visita el algoritmo.

In [6]:
def minimax_contando(nodo, es_max, arbol, utilidades, contador):
    contador["visitados"] += 1

    if nodo in utilidades:
        contador["terminales"] += 1
        return utilidades[nodo]

    valores = [
        minimax_contando(hijo, not es_max, arbol, utilidades, contador)
        for hijo in arbol[nodo]
    ]

    return max(valores) if es_max else min(valores)


contador = {"visitados": 0, "terminales": 0}
valor = minimax_contando(
    "A",
    True,
    arbol_profundo,
    utilidades_profundo,
    contador
)

print("Valor Minimax:", valor)
print("Nodos visitados:", contador["visitados"])
print("Hojas evaluadas:", contador["terminales"])

Valor Minimax: 5
Nodos visitados: 15
Hojas evaluadas: 8


# 8. Caso aplicado: juego de las piedras

Trabajaremos con un juego muy sencillo:

- Existe una pila con cierta cantidad de piedras.
- En cada turno un jugador puede retirar `1`, `2` o `3` piedras.
- El jugador que retira la **última piedra gana**.

Representaremos un estado como:

```python
(piedras_restantes, jugador)
```

donde:

- `jugador = 1` representa a MAX;
- `jugador = -1` representa a MIN.

In [7]:
MOVIMIENTOS = (1, 2, 3)

def movimientos_validos(piedras):
    return [m for m in MOVIMIENTOS if m <= piedras]


for n in range(1, 8):
    print(n, "piedras ->", movimientos_validos(n))

1 piedras -> [1]
2 piedras -> [1, 2]
3 piedras -> [1, 2, 3]
4 piedras -> [1, 2, 3]
5 piedras -> [1, 2, 3]
6 piedras -> [1, 2, 3]
7 piedras -> [1, 2, 3]


## 8.1 Utilidad del estado terminal

Cuando no quedan piedras, significa que el jugador anterior tomó la última.

Si el jugador que debe mover ahora es MAX, entonces MIN realizó la jugada anterior
y ganó. Por tanto, la utilidad para MAX es `-1`.

Si debe mover MIN, MAX realizó la jugada anterior y ganó. La utilidad es `+1`.

In [8]:
def minimax_piedras(piedras, turno_max):
    if piedras == 0:
        return -1 if turno_max else 1

    valores = []

    for retirar in movimientos_validos(piedras):
        valor = minimax_piedras(
            piedras - retirar,
            not turno_max
        )
        valores.append(valor)

    return max(valores) if turno_max else min(valores)


for piedras in range(1, 11):
    print(
        f"{piedras:2d} piedras -> valor Minimax:",
        minimax_piedras(piedras, True)
    )

 1 piedras -> valor Minimax: 1
 2 piedras -> valor Minimax: 1
 3 piedras -> valor Minimax: 1
 4 piedras -> valor Minimax: -1
 5 piedras -> valor Minimax: 1
 6 piedras -> valor Minimax: 1
 7 piedras -> valor Minimax: 1
 8 piedras -> valor Minimax: -1
 9 piedras -> valor Minimax: 1
10 piedras -> valor Minimax: 1


## 8.2 Encontrar la mejor jugada

Ahora determinaremos cuántas piedras debería retirar MAX.

In [9]:
def mejor_movimiento_piedras(piedras):
    opciones = []

    for retirar in movimientos_validos(piedras):
        valor = minimax_piedras(piedras - retirar, False)
        opciones.append((valor, retirar))

    mejor_valor, mejor_movimiento = max(opciones, key=lambda x: x[0])

    return {
        "retirar": mejor_movimiento,
        "valor": mejor_valor,
        "opciones": opciones,
    }


for piedras in range(1, 11):
    print(
        f"{piedras:2d} piedras ->",
        mejor_movimiento_piedras(piedras)
    )

 1 piedras -> {'retirar': 1, 'valor': 1, 'opciones': [(1, 1)]}
 2 piedras -> {'retirar': 2, 'valor': 1, 'opciones': [(-1, 1), (1, 2)]}
 3 piedras -> {'retirar': 3, 'valor': 1, 'opciones': [(-1, 1), (-1, 2), (1, 3)]}
 4 piedras -> {'retirar': 1, 'valor': -1, 'opciones': [(-1, 1), (-1, 2), (-1, 3)]}
 5 piedras -> {'retirar': 1, 'valor': 1, 'opciones': [(1, 1), (-1, 2), (-1, 3)]}
 6 piedras -> {'retirar': 2, 'valor': 1, 'opciones': [(-1, 1), (1, 2), (-1, 3)]}
 7 piedras -> {'retirar': 3, 'valor': 1, 'opciones': [(-1, 1), (-1, 2), (1, 3)]}
 8 piedras -> {'retirar': 1, 'valor': -1, 'opciones': [(-1, 1), (-1, 2), (-1, 3)]}
 9 piedras -> {'retirar': 1, 'valor': 1, 'opciones': [(1, 1), (-1, 2), (-1, 3)]}
10 piedras -> {'retirar': 2, 'valor': 1, 'opciones': [(-1, 1), (1, 2), (-1, 3)]}


### Preguntas de análisis

1. ¿Qué cantidades iniciales de piedras representan una posición desfavorable para MAX?
2. ¿Existe algún patrón?
3. ¿Por qué algunas posiciones tienen valor `-1` incluso si MAX todavía dispone de varios movimientos?
4. ¿Puede haber más de una jugada igualmente buena?
5. ¿Qué cambiaría si fuera obligatorio retirar únicamente `1` o `2` piedras?

## 9 Minimax con profundidad limitada

La siguiente versión admite:

- una profundidad máxima;
- una función de evaluación para estados no terminales.

Este esquema es mucho más cercano al utilizado en juegos reales.

In [10]:
def minimax_limitado(
    estado,
    profundidad,
    es_max,
    es_terminal,
    utilidad,
    sucesores,
    evaluar
):
    if es_terminal(estado):
        return utilidad(estado)

    if profundidad == 0:
        return evaluar(estado)

    valores = [
        minimax_limitado(
            hijo,
            profundidad - 1,
            not es_max,
            es_terminal,
            utilidad,
            sucesores,
            evaluar
        )
        for hijo in sucesores(estado)
    ]

    return max(valores) if es_max else min(valores)

# 10. Taller

Implemente Minimax para **Tres en raya (Tic-Tac-Toe)**.

Puede representar el tablero como una tupla de nueve posiciones:

```python
(
    "X", "O", " ",
    " ", "X", " ",
    "O", " ", " "
)
```

Suponga:

- `X` es MAX;
- `O` es MIN;
- victoria de `X`: `+1`;
- empate: `0`;
- victoria de `O`: `-1`.

Implemente como mínimo:

```python
acciones(tablero)
resultado(tablero, accion, jugador)
terminal(tablero)
utilidad(tablero)
minimax_tictactoe(tablero, es_max)
```

In [11]:
LINEAS = [
    (0, 1, 2), (3, 4, 5), (6, 7, 8),   # filas
    (0, 3, 6), (1, 4, 7), (2, 5, 8),   # columnas
    (0, 4, 8), (2, 4, 6),              # diagonales
]

def acciones(tablero):
    return [i for i in range(9) if tablero[i] == " "]

def resultado(tablero, accion, jugador):
    nuevo = list(tablero)
    nuevo[accion] = jugador
    return tuple(nuevo)

def ganador(tablero):
    for a, b, c in LINEAS:
        if tablero[a] != " " and tablero[a] == tablero[b] == tablero[c]:
            return tablero[a]
    return None

def terminal(tablero):
    return ganador(tablero) is not None or " " not in tablero

def utilidad(tablero):
    g = ganador(tablero)
    if g == "X":
        return 1
    if g == "O":
        return -1
    return 0

def minimax_tictactoe(tablero, es_max):
    if terminal(tablero):
        return utilidad(tablero)
    jugador = "X" if es_max else "O"
    valores = []
    for accion in acciones(tablero):
        siguiente = resultado(tablero, accion, jugador)
        valores.append(minimax_tictactoe(siguiente, not es_max))
    return max(valores) if es_max else min(valores)

### Para el juego de 3 en raya:

Construya un árbol de al menos tres niveles y:

1. asigne valores de utilidad a las hojas;
2. calcule manualmente los valores Minimax;
3. compruebe el resultado con Python;
4. indique la jugada elegida por MAX.

Complete la implementación propuesta y pruebe diferentes tableros.

### Para el juego de las piedras
Modifique las reglas para permitir retirar únicamente `1`, `2` o `4` piedras.

Analice:

- posiciones ganadoras;
- posiciones perdedoras;
- mejor movimiento para MAX.


### Pregunta final

**¿Por qué una decisión que parece buena de manera inmediata puede ser mala después de considerar la respuesta del adversario?**

### Uso de IA generativa

Si utiliza IA generativa, indique:

- herramienta utilizada;
- propósito de uso;
- partes de la solución en las que fue empleada.

## Pruebas de la implementación

Antes de armar el árbol de 3 niveles, pruebo el minimax con un tablero fácil para asegurarme de que las funciones estén bien.

Hago un pequeño helper para imprimir el tablero (las casillas vacías las muestro como `.`).

In [12]:
def imprimir_tablero(t):
    for i in range(0, 9, 3):
        celdas = []
        for x in t[i:i+3]:
            if x == " ":
                celdas.append(".")
            else:
                celdas.append(x)
        print(" | ".join(celdas))
        if i < 6:
            print("---------")

### Caso 1: X puede ganar en el siguiente movimiento

MAX (X) tiene dos fichas en la primera fila. La jugada obvia es completar la línea en la casilla 2. El valor Minimax debería ser `+1`.

In [13]:
tablero_1 = (
    "X", "X", " ",
    "O", "O", " ",
    " ", " ", " ",
)

imprimir_tablero(tablero_1)
print()
print("Valor Minimax (turno X):", minimax_tictactoe(tablero_1, True))

X | X | .
---------
O | O | .
---------
. | . | .

Valor Minimax (turno X): 1


In [14]:
def mejor_movimiento_tictactoe(tablero, es_max):
    jugador = "X" if es_max else "O"
    opciones = []
    for accion in acciones(tablero):
        siguiente = resultado(tablero, accion, jugador)
        valor = minimax_tictactoe(siguiente, not es_max)
        opciones.append((valor, accion))
    if es_max:
        return max(opciones, key=lambda x: x[0])
    return min(opciones, key=lambda x: x[0])


valor, accion = mejor_movimiento_tictactoe(tablero_1, True)
print(f"Mejor jugada para X: casilla {accion} (valor {valor})")

Mejor jugada para X: casilla 2 (valor 1)


### Caso 2: árbol de 3 niveles

Ahora un tablero con más jugadas hechas. Turno de X (MAX):

```
X | O | .
---------
. | X | .
---------
O | . | .
```

Las casillas libres son 2, 3, 5, 7 y 8. X ya tiene la diagonal 0-4 abierta, así que mi intuición es que va a jugar 8 para cerrarla. Corro el algoritmo a ver.

In [15]:
tablero_2 = (
    "X", "O", " ",
    " ", "X", " ",
    "O", " ", " ",
)

imprimir_tablero(tablero_2)
print()
valor, accion = mejor_movimiento_tictactoe(tablero_2, True)
print(f"Valor Minimax: {valor}")
print(f"Jugada elegida por X: casilla {accion}")

X | O | .
---------
. | X | .
---------
O | . | .

Valor Minimax: 1
Jugada elegida por X: casilla 3


**Análisis.** El algoritmo confirma que X gana desde aquí (valor `+1`) y elige la casilla `8`, cerrando la diagonal 0-4-8. Coincide con lo que había pensado: MIN no puede impedir la línea porque X ya tiene 0 y 4, y MIN solo tiene un movimiento por turno.

## Variante del juego de piedras: movimientos {1, 2, 4}

Cambio las reglas: en cada turno se pueden retirar `1`, `2` o `4` piedras (ya no `3`). Miro qué pasa con el patrón de posiciones ganadoras y perdedoras para MAX.

In [16]:
def movimientos_validos_124(piedras):
    return [m for m in (1, 2, 4) if m <= piedras]

def minimax_piedras_124(piedras, turno_max):
    if piedras == 0:
        return -1 if turno_max else 1
    valores = []
    for retirar in movimientos_validos_124(piedras):
        valor = minimax_piedras_124(piedras - retirar, not turno_max)
        valores.append(valor)
    return max(valores) if turno_max else min(valores)


print(f"{'Piedras':>8}{'Valor':>8}{'Retirar':>10}")
for piedras in range(1, 16):
    valor = minimax_piedras_124(piedras, True)
    # el primer movimiento que consigue el valor optimo
    mejor = None
    for retirar in movimientos_validos_124(piedras):
        v = minimax_piedras_124(piedras - retirar, False)
        if v == valor:
            mejor = retirar
            break
    print(f"{piedras:>8}{valor:>8}{str(mejor):>10}")

 Piedras   Valor   Retirar
       1       1         1
       2       1         2
       3      -1         1
       4       1         1
       5       1         2
       6      -1         1
       7       1         1
       8       1         2
       9      -1         1
      10       1         1
      11       1         2
      12      -1         1
      13       1         1
      14       1         2
      15      -1         1


**Análisis.** Con los movimientos originales `{1, 2, 3}` los perdedores para MAX eran los múltiplos de 4 (4, 8, 12, ...).

Aquí con `{1, 2, 4}` los perdedores son 3, 6, 9, 12 y 15 — o sea, múltiplos de 3. El patrón cambia por completo al cambiar los movimientos permitidos: antes había que evitar dejarle al otro múltiplos de 4, ahora hay que evitar dejarle múltiplos de 3.

## Pregunta final

**¿Por qué una decisión que parece buena de manera inmediata puede ser mala después de considerar la respuesta del adversario?**

Porque después de mi jugada el rival también juega, y va a elegir la respuesta que peor me deje. Si yo miro solo el resultado inmediato, no estoy contando con eso.

El árbol de la sección 3 es un buen ejemplo. La hoja más alta es `9`, bajo C. A primera vista C parece la mejor rama. Pero MIN juega después de MAX, y bajo C hay una hoja de valor `1` (C2). MIN me va a llevar a esa, no al 9. Así que ir por C me deja con un 1. En cambio, ir por D me deja con un 6 en el peor caso.

Por eso Minimax mira todo el árbol asumiendo que el rival juega perfecto: no importa qué tan bueno se ve un estado si el rival me puede llevar a otro peor.

## Uso de IA generativa

- **Herramienta utilizada:** Claude (Anthropic).
- **Propósito de uso:** apoyo en la redacción y revisión de estilo de la respuesta a la pregunta final.
- **Partes en las que fue empleada:** pregunta final del taller.